In [1]:
!pip install torch torchvision torchaudio --quiet

In [2]:
!pip install numpy pandas scikit-learn matplotlib --quiet

In [3]:
import os
import math
import time
import random
import warnings
import csv

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import Adam

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings("ignore")

In [4]:
DATA_PATH    = "clean_daily_close.csv" # Adjust as needed
OUTPUT_CSV   = "standalone_lstm_results.csv"
TARGET_COL   = "Close"

LOOK_BACK    = 20
PRED_LEN     = 1

# Hyperparameters (FIXED)
LSTM_LAYERS  = 4
HIDDEN_UNITS = 4
DROPOUT      = 0.1
EPOCHS       = 100
BATCH_SIZE   = 32
LR           = 0.001

N_EXPERIMENTS      = 50
ROLLING_VOL_WINDOW = 30
MULTISTEP_HORIZONS = [5, 21]

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [5]:
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False

In [6]:
# DATA SPLITTING & METRICS
# ============================================================
def compute_metrics(y_true, y_pred):
    y_true = np.array(y_true, dtype=np.float64)
    y_pred = np.array(y_pred, dtype=np.float64)
    mae  = float(mean_absolute_error(y_true, y_pred))
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))

    mask = np.abs(y_true) > 1e-8
    mape = float(np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100)
    r2   = float(r2_score(y_true, y_pred))
    return {"MAE": mae, "RMSE": rmse, "MAPE": mape, "R2": r2}

def split_data(df):
    """Chronologically split the dataframe."""
    df = df.sort_values("Date").reset_index(drop=True)

    train_mask = df["Date"].dt.year <= 2022
    test_mask  = df["Date"].dt.year >= 2023

    df_train_full = df[train_mask].reset_index(drop=True)
    df_test       = df[test_mask].reset_index(drop=True)

    val_size = int(len(df_train_full) * 0.10)
    df_train = df_train_full.iloc[:-val_size].reset_index(drop=True)
    df_val   = df_train_full.iloc[-val_size:].reset_index(drop=True)

    return df_train, df_val, df_test

In [7]:
def create_windows(series, look_back):
    """Creates overlapping windows: X=(N, look_back, 1), y=(N, 1)"""
    X, y = [], []
    for i in range(len(series) - look_back):
        X.append(series[i : i + look_back])
        y.append(series[i + look_back])
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.float32)

class TSDBDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.from_numpy(X)
        self.y = torch.from_numpy(y)
    def __len__(self):
        return len(self.y)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [8]:
class StandaloneLSTM(nn.Module):
    def __init__(self, input_size=1, hidden_size=HIDDEN_UNITS, num_layers=LSTM_LAYERS, dropout=DROPOUT):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0
        )
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        # x shape: (Batch, Seq_len, Features)
        out, _ = self.lstm(x)
        # return the last time step's output
        return self.fc(out[:, -1, :])

In [9]:
def recursive_multi_step_forecast(model, seed_window_scaled, steps, scaler):
    """
    Forecasting `steps` ahead recursively. No ground truth is injected during the steps.
    seed_window_scaled: shape (look_back, 1), the last `look_back` values of Validation.
    """
    model.eval()
    curr_window = torch.tensor(seed_window_scaled, dtype=torch.float32).unsqueeze(0).to(DEVICE) # (1, look_back, 1)

    preds_scaled = []
    with torch.no_grad():
        for _ in range(steps):
            pred = model(curr_window) # (1, 1)
            preds_scaled.append(pred.item())

            # Slide window
            pred_tensor = pred.unsqueeze(1) # (1, 1, 1)
            curr_window = torch.cat((curr_window[:, 1:, :], pred_tensor), dim=1)

    # Inverse transform predictions
    preds_orig = scaler.inverse_transform(np.array(preds_scaled).reshape(-1, 1)).flatten()
    return preds_orig

In [10]:

def display_volatility_regime(df_test, y_true, y_hat):
    close_all = df_test[TARGET_COL].values.astype(np.float64)
    # Align log returns with close_all length by prepending a NaN
    log_ret_vals = np.append([np.nan], np.log(close_all[1:] / close_all[:-1]))
    rol_vol = pd.Series(log_ret_vals).rolling(ROLLING_VOL_WINDOW).std()

    # In this Standalone script, test predictions are padded so len(y_true) == len(df_test)
    # We clip strictly to the minimum length to guarantee no index errors
    min_len = min(len(y_true), len(rol_vol))

    vol_vals = rol_vol.values[:min_len]
    y_t = y_true[:min_len]
    y_h = y_hat[:min_len]

    vol_med = float(np.nanmedian(vol_vals))
    vol_vals = np.where(np.isnan(vol_vals), vol_med, vol_vals)

    high_mask = vol_vals >= vol_med
    low_mask  = ~high_mask

    if high_mask.sum() > 0 and low_mask.sum() > 0:
        m_h = compute_metrics(y_t[high_mask], y_h[high_mask])
        m_l = compute_metrics(y_t[low_mask],  y_h[low_mask])

        print("\n" + "=" * 66)
        print("  VOLATILITY REGIME VALIDATION (Seed 50)")
        print(f"  30-day rolling vol | Median = {vol_med:.6f}")
        print("=" * 66)
        print(f'  {"Metric":<12} {"High Volatility":>18} {"Low Volatility":>18}')
        print("-" * 66)
        for k in ["MAE", "RMSE", "MAPE", "R2"]:
            print(f"  {k:<12} {m_h[k]:>18.4f} {m_l[k]:>18.4f}")
        print("=" * 66)
        print(f"  High-vol days : {high_mask.sum()}")
        print(f"  Low-vol  days : {low_mask.sum()}")


In [11]:
def run_experiment(df_train, df_val, df_test, seed):
    set_seed(seed)
    t0 = time.time()

    # Preprocessing (MinMaxScaler fitted ONLY on training)
    scaler = MinMaxScaler()
    train_scaled = scaler.fit_transform(df_train[[TARGET_COL]].values)
    val_scaled   = scaler.transform(df_val[[TARGET_COL]].values)
    test_scaled  = scaler.transform(df_test[[TARGET_COL]].values)

    # Padding sequences for validation/test without leakage
    val_context  = np.concatenate((train_scaled[-LOOK_BACK:], val_scaled))
    test_context = np.concatenate((val_scaled[-LOOK_BACK:], test_scaled))

    X_tr, y_tr = create_windows(train_scaled, LOOK_BACK)
    X_vl, y_vl = create_windows(val_context, LOOK_BACK)
    X_te, y_te = create_windows(test_context, LOOK_BACK)

    train_loader = DataLoader(
        TSDBDataset(X_tr, y_tr), batch_size=BATCH_SIZE, shuffle=False
    )

    model = StandaloneLSTM().to(DEVICE)
    optimizer = Adam(model.parameters(), lr=LR)
    criterion = nn.MSELoss()

    # Training Loop
    model.train()
    for _ in range(EPOCHS):
        for bx, by in train_loader:
            bx, by = bx.to(DEVICE), by.to(DEVICE)
            optimizer.zero_grad()

            # by is shape (B, 1), pred is shape (B, 1)
            pred = model(bx)
            loss = criterion(pred, by)

            loss.backward()
            optimizer.step()

    t_elapsed = time.time() - t0

    # Point-Forecast on Test set
    model.eval()
    y_hat_scaled = []
    test_loader = DataLoader(TSDBDataset(X_te, y_te), batch_size=BATCH_SIZE, shuffle=False)

    with torch.no_grad():
        for bx, _ in test_loader:
            bx = bx.to(DEVICE)
            pred = model(bx)
            y_hat_scaled.append(pred.cpu().numpy())

    y_hat_scaled = np.concatenate(y_hat_scaled)
    y_hat  = scaler.inverse_transform(y_hat_scaled).flatten()
    y_true = scaler.inverse_transform(y_te).flatten()

    # Standard metrics
    metrics = compute_metrics(y_true, y_hat)
    metrics["training_time_sec"] = t_elapsed

    # Robustness metrics: Multi-step forecasting
    # Seed the recursive logic using the VERY LAST look_back values from the Validation set
    seed_window = val_context[-LOOK_BACK:]
    for step_count in MULTISTEP_HORIZONS:
        # Forecast recursively
        multistep_preds = recursive_multi_step_forecast(model, seed_window, step_count, scaler)
        # Compare against the first `step_count` values of the true test set
        mstep_true = scaler.inverse_transform(test_scaled[:step_count]).flatten()
        mstep_metrics = compute_metrics(mstep_true, multistep_preds)

        # Add to metrics dict for tracking
        metrics[f"multi_{step_count}d_MAE"] = mstep_metrics["MAE"]
        metrics[f"multi_{step_count}d_RMSE"] = mstep_metrics["RMSE"]

    return metrics, y_true, y_hat


In [12]:
def main():
    print(f"Device    : {DEVICE}")
    print(f"PyTorch   : {torch.__version__}")
    print(f"Model     : Standalone LSTM ({LSTM_LAYERS} layers, {HIDDEN_UNITS} hidden, 1 output)")
    print(f"Target    : {TARGET_COL} (Only)")
    print(f"Split     : 2015-2022 (Train/Val), 2023-2025 (Test)")
    print(f"Runs      : {N_EXPERIMENTS}")
    print("=" * 65)

    df = pd.read_csv(DATA_PATH, parse_dates=["Date"])
    df_train, df_val, df_test = split_data(df)

    print(f"Train / Val / Test shapes: {df_train.shape[0]} / {df_val.shape[0]} / {df_test.shape[0]}")
    print("=" * 65)

    all_results = []
    _last_y_true = None
    _last_y_hat  = None

    for seed in range(1, N_EXPERIMENTS + 1):
        print(f"[{seed:02d}/{N_EXPERIMENTS}] Executing seed={seed}...", end=" ", flush=True)
        m, yt, yp = run_experiment(df_train, df_val, df_test, seed=seed)
        m["experiment"] = seed
        m["seed"]       = seed
        all_results.append(m)

        print(f"Done | MAE={m['MAE']:8.2f} | R²={m['R2']:7.4f} | Time={m['training_time_sec']:.1f}s", flush=True)

        _last_y_true = yt
        _last_y_hat  = yp

    print("\n✅ All 50 experiments complete.")

    # Aggregate Metrics
    keys = ["MAE", "RMSE", "MAPE", "R2", "training_time_sec"]
    for d in MULTISTEP_HORIZONS:
        keys.extend([f"multi_{d}d_MAE", f"multi_{d}d_RMSE"])

    print("\n" + "=" * 80)
    print("  STANDALONE LSTM — 50-RUN STATISTICAL SUMMARY")
    print("=" * 80)
    print(f'  {"Metric":<20} {"Mean":>12} {"± Std":>12} {"Min":>10} {"Max":>10}')
    print("-" * 80)

    for k in keys:
        arr = [r[k] for r in all_results]
        mn, sd, mi, mx = np.mean(arr), np.std(arr), np.min(arr), np.max(arr)

        # formatting adjustments
        name = "Time (s)" if k == "training_time_sec" else k
        print(f"  {name:<20} {mn:>12.4f} {sd:>12.4f} {mi:>10.4f} {mx:>10.4f}")

    print("=" * 80)

    # Save Results
    with open(OUTPUT_CSV, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=["experiment", "seed"] + keys)
        writer.writeheader()
        writer.writerows(all_results)
    print(f"Results saved to: {OUTPUT_CSV}")

    # Print the Recursive Volatility logic results for the final seed (seed=50)
    display_volatility_regime(df_test, _last_y_true, _last_y_hat)


if __name__ == "__main__":
    main()


Device    : cuda
PyTorch   : 2.10.0+cu128
Model     : Standalone LSTM (4 layers, 4 hidden, 1 output)
Target    : Close (Only)
Split     : 2015-2022 (Train/Val), 2023-2025 (Test)
Runs      : 50
Train / Val / Test shapes: 1770 / 196 / 633
[01/50] Executing seed=1... Done | MAE= 4297.15 | R²=-2.6731 | Time=25.3s
[02/50] Executing seed=2... Done | MAE= 3657.11 | R²=-1.7630 | Time=19.6s
[03/50] Executing seed=3... Done | MAE= 3895.87 | R²=-2.1123 | Time=19.5s
[04/50] Executing seed=4... Done | MAE= 4316.81 | R²=-2.6745 | Time=20.9s
[05/50] Executing seed=5... Done | MAE= 3630.61 | R²=-1.7304 | Time=19.0s
[06/50] Executing seed=6... Done | MAE= 4324.79 | R²=-2.7096 | Time=19.4s
[07/50] Executing seed=7... Done | MAE= 4383.99 | R²=-2.8228 | Time=19.6s
[08/50] Executing seed=8... Done | MAE= 3418.93 | R²=-1.4290 | Time=18.8s
[09/50] Executing seed=9... Done | MAE= 3983.33 | R²=-2.2556 | Time=19.7s
[10/50] Executing seed=10... Done | MAE= 3637.73 | R²=-1.7136 | Time=18.9s
[11/50] Executing seed